# Notebook de Adquisición — Siniestralidad Vial Colombia
**Minería de Datos · Corte 1 · 29 de septiembre de 2026**

**Integrantes:**
- Andrés Felipe Patiño
- Bryan Bedoya Calderón
- Juan Camilo Arenas Villa

**Repositorio:** https://github.com/JuanCamilo620/siniestralidad-vial-colombia

---

## Objetivo

Leer, cruzar y documentar las dos fuentes de datos del proyecto integrador:
1. **Sectores críticos de siniestralidad vial** (Datos Abiertos Colombia, ANSV) — 2015–2019
2. **Personas fallecidas en siniestros viales** (Observatorio Nacional de Seguridad Vial) — 2021–2024 (certificado) + 2025 (preliminar)

El cruce se hará a nivel de **Departamento + Municipio**, porque las fuentes están a niveles distintos: una tiene una fila por tramo de carretera, la otra tiene una fila por persona fallecida.

In [29]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import unicodedata

print("Ambiente listo")
print("Versión de pandas:", pd.__version__)

Ambiente listo
Versión de pandas: 2.2.3


## 1. Subir los archivos

Vamos a subir los 3 archivos al entorno de Colab:
- `SECTORES_CRITICOS_DE_SINIESTRALIDAD_VIAL_20260912.csv` (Fuente 1)
- `versionfinal.csv` (Fuente 2, 2021–2024 certificado)
- `version_parcial_a_diciembre.csv` (Fuente 2, 2021–2025, incluye 2025 preliminar)

**Ojo:** Los archivos subidos en Colab se borran al cerrar la sesión. Si vuelven otro día, hay que subirlos de nuevo.

In [30]:
from google.colab import files

print("Sube los 3 archivos CSV:")
subidos = files.upload()

# Ver qué archivos se subieron
print("\nArchivos subidos:")
for nombre in subidos.keys():
    print(f"  - {nombre}")

Sube los 3 archivos CSV:


Saving SECTORES_CRITICOS_DE_SINIESTRALIDAD_VIAL_20260912.csv to SECTORES_CRITICOS_DE_SINIESTRALIDAD_VIAL_20260912 (2).csv
Saving version_parcial_a_diciembre.csv to version_parcial_a_diciembre (1).csv
Saving versionfinal.csv to versionfinal (1).csv

Archivos subidos:
  - SECTORES_CRITICOS_DE_SINIESTRALIDAD_VIAL_20260912 (2).csv
  - version_parcial_a_diciembre (1).csv
  - versionfinal (1).csv


## 2. Lectura de la Fuente 1 — Sectores Críticos

**Fuente 1:** `SECTORES_CRITICOS_DE_SINIESTRALIDAD_VIAL_20260912.csv`
- Separador: **coma (`,`)**
- Una fila = un tramo de carretera
- Período: 2015–2019
- 316 filas, 13 columnas

In [31]:
# Leer Fuente 1 (separador coma)
sectores = pd.read_csv("SECTORES_CRITICOS_DE_SINIESTRALIDAD_VIAL_20260912.csv")

# Verificación obligatoria
print("=" * 60)
print("FUENTE 1 — SECTORES CRÍTICOS")
print("=" * 60)
print(f"\nShape (filas, columnas): {sectores.shape}")
print(f"\nPrimeras 5 filas:")
print(sectores.head())
print(f"\nTipos de datos:")
print(sectores.dtypes)

FUENTE 1 — SECTORES CRÍTICOS

Shape (filas, columnas): (316, 13)

Primeras 5 filas:
    ID_MT       ENTIDAD  GiZScore  Fallecidos  GiPValue  \
0   4G013           ANI  1.771262          11  0.076517   
1   4G013           ANI  3.818008          23  0.000135   
2   4G013           ANI  2.796662           1  0.005163   
3   4G013           ANI  1.725132           1  0.084504   
4  <Null>  INVIAS-OTROS  1.725132          13  0.084504   

                     Tramo                         Nombre   Latitud  \
0   Bogotá - Villavicencio  IP  –  Chirajara - Fundadores  4.452987   
1   Bogotá - Villavicencio  IP  –  Chirajara - Fundadores  4.228251   
2   Bogotá - Villavicencio  IP  –  Chirajara - Fundadores  4.203601   
3   Bogotá - Villavicencio  IP  –  Chirajara - Fundadores  4.121453   
4  Granada - Villavicencio                            NaN  4.115757   

    Longitud  PR      Municipio  Departamento  divipola  
0 -74.046677  10       CHIPAQUE  CUNDINAMARCA     25178  
1 -73.818342  58  

**Observación de la Fuente 1:** Confirmamos que tiene 316 filas y 13 columnas. Las columnas geográficas clave para el cruce son `Departamento` y `Municipio`. También tiene `Fallecidos`, que es la variable de interés.

## 3. Lectura de la Fuente 2 — Víctimas Fatales

**Fuente 2:** `versionfinal.csv` (2021–2024, certificado)
- Separador: **punto y coma (`;`)**
- Una fila = una persona fallecida
- Período: 2021–2024
- 28 columnas

In [32]:
# Leer Fuente 2 (separador punto y coma)
fallecidos = pd.read_csv("versionfinal.csv", sep=";")

# Verificación obligatoria
print("=" * 60)
print("FUENTE 2 — VÍCTIMAS FATALES (2021–2024 certificado)")
print("=" * 60)
print(f"\nShape (filas, columnas): {fallecidos.shape}")
print(f"\nPrimeras 5 filas:")
print(fallecidos.head())
print(f"\nTipos de datos:")
print(fallecidos.dtypes)

FUENTE 2 — VÍCTIMAS FATALES (2021–2024 certificado)

Shape (filas, columnas): (32832, 28)

Primeras 5 filas:
  Departamento   Municipio EstadoVictima  AnoHecho MesOCurrencia  \
0      QUINDIO     ARMENIA       MUERTOS      2024    02.FEBRERO   
1    ANTIOQUIA    MEDELLIN       MUERTOS      2021      07.JULIO   
2      BOLIVAR  SANTA ROSA       MUERTOS      2023  11.NOVIEMBRE   
3    ANTIOQUIA    MEDELLIN       MUERTOS      2023  11.NOVIEMBRE   
4    MAGDALENA    EL BANCO       MUERTOS      2023    02.FEBRERO   

  DiaOcurrencia TipoVehiculoGrupo TipoServicio CondicionVictima  \
0       1.LUNES       MOTOCICLETA   PARTICULAR        CONDUCTOR   
1      6.SÁBADO         NO APLICA    NO APLICA           PEATÓN   
2      2.MARTES       MOTOCICLETA   PARTICULAR        CONDUCTOR   
3     7.DOMINGO         BICICLETA    NO APLICA        CONDUCTOR   
4      2.MARTES         BICICLETA    NO APLICA        CONDUCTOR   

              ActorVial  ...        EstadoVia ActividadVictima  CausaMuerte  \


In [33]:
# Leer Fuente 2 preliminar (incluye 2025)
fallecidos_2025 = pd.read_csv("version_parcial_a_diciembre.csv", sep=";")

print("=" * 60)
print("FUENTE 2 — VÍCTIMAS FATALES (2021–2025 preliminar)")
print("=" * 60)
print(f"\nShape (filas, columnas): {fallecidos_2025.shape}")
print(f"\nAños disponibles:")
print(fallecidos_2025["AnoHecho"].value_counts().sort_index())

FUENTE 2 — VÍCTIMAS FATALES (2021–2025 preliminar)

Shape (filas, columnas): (40845, 28)

Años disponibles:
AnoHecho
2021    7258
2022    8255
2023    8390
2024    8255
2025    8687
Name: count, dtype: int64


**Decisión:** Usaremos `versionfinal.csv` (2021–2024 certificado) como base principal. La versión preliminar se usará solo para verificar tendencias de 2025, sin incorporarla al análisis principal porque está sujeta a revisión.

## 4. Normalización de texto

Antes de cruzar, hay que normalizar los nombres de Departamento y Municipio en **ambas fuentes**, porque pueden tener:
- Diferencias de mayúsculas/minúsculas (`CUNDINAMARCA` vs `Cundinamarca`)
- Tildes (`BOGOTÁ` vs `BOGOTA`)
- Espacios extra al inicio o final

Sin esto, el merge falla silenciosamente y pierde filas.

In [34]:
def normalizar_texto(texto):
    """
    Normaliza un texto para que sea comparable entre fuentes:
    - Convierte a mayúsculas
    - Quita tildes
    - Quita espacios extra
    """
    if pd.isna(texto):
        return texto
    # Convertir a string, mayúsculas y quitar espacios
    texto = str(texto).strip().upper()
    # Quitar tildes
    texto = ''.join(
        c for c in unicodedata.normalize('NFKD', texto)
        if not unicodedata.combining(c)
    )
    return texto

# Probar la función
print(normalizar_texto("  Bogotá D.C.  "))
print(normalizar_texto("CUNDINAMARCA"))
print(normalizar_texto("Medellín"))

BOGOTA D.C.
CUNDINAMARCA
MEDELLIN


In [35]:
# Normalizar en Fuente 1
sectores["Departamento_norm"] = sectores["Departamento"].apply(normalizar_texto)
sectores["Municipio_norm"] = sectores["Municipio"].apply(normalizar_texto)

# Normalizar en Fuente 2
fallecidos["Departamento_norm"] = fallecidos["Departamento"].apply(normalizar_texto)
fallecidos["Municipio_norm"] = fallecidos["Municipio"].apply(normalizar_texto)

print("Normalización aplicada.")
print("\nEjemplo Fuente 1:")
print(sectores[["Departamento", "Departamento_norm", "Municipio", "Municipio_norm"]].head())
print("\nEjemplo Fuente 2:")
print(fallecidos[["Departamento", "Departamento_norm", "Municipio", "Municipio_norm"]].head())

Normalización aplicada.

Ejemplo Fuente 1:
   Departamento Departamento_norm      Municipio Municipio_norm
0  CUNDINAMARCA      CUNDINAMARCA       CHIPAQUE       CHIPAQUE
1  CUNDINAMARCA      CUNDINAMARCA     GUAYABETAL     GUAYABETAL
2  CUNDINAMARCA      CUNDINAMARCA     GUAYABETAL     GUAYABETAL
3          META              META  VILLAVICENCIO  VILLAVICENCIO
4          META              META  VILLAVICENCIO  VILLAVICENCIO

Ejemplo Fuente 2:
  Departamento Departamento_norm   Municipio Municipio_norm
0      QUINDIO           QUINDIO     ARMENIA        ARMENIA
1    ANTIOQUIA         ANTIOQUIA    MEDELLIN       MEDELLIN
2      BOLIVAR           BOLIVAR  SANTA ROSA     SANTA ROSA
3    ANTIOQUIA         ANTIOQUIA    MEDELLIN       MEDELLIN
4    MAGDALENA         MAGDALENA    EL BANCO       EL BANCO


## 5. Agrupación de la Fuente 2

**Problema:** La Fuente 1 tiene una fila por tramo y la Fuente 2 tiene una fila por persona fallecida. No se pueden cruzar directo.

**Solución:** Agrupar la Fuente 2 por Departamento + Municipio para contar cuántas personas fallecieron en cada municipio. Ahora ambas fuentes están al mismo nivel (una fila por municipio).

In [36]:
# Agrupar fallecidos por Departamento + Municipio
fallecidos_por_municipio = (
    fallecidos
    .groupby(["Departamento_norm", "Municipio_norm"])
    .size()
    .reset_index(name="total_fallecidos")
)

print("=" * 60)
print("FALLECIDOS AGRUPADOS POR MUNICIPIO")
print("=" * 60)
print(f"Municipios únicos con fallecidos: {len(fallecidos_por_municipio)}")
print(f"\nPrimeras 10 filas:")
print(fallecidos_por_municipio.head(10))
print(f"\nTop 10 municipios con más fallecidos:")
print(fallecidos_por_municipio.sort_values("total_fallecidos", ascending=False).head(10))

FALLECIDOS AGRUPADOS POR MUNICIPIO
Municipios únicos con fallecidos: 1006

Primeras 10 filas:
  Departamento_norm Municipio_norm  total_fallecidos
0          AMAZONAS        LETICIA                13
1         ANTIOQUIA      ABEJORRAL                 6
2         ANTIOQUIA          AMAGA                33
3         ANTIOQUIA         AMALFI                 8
4         ANTIOQUIA          ANDES                26
5         ANTIOQUIA    ANGELOPOLIS                 2
6         ANTIOQUIA      ANGOSTURA                 1
7         ANTIOQUIA          ANORI                 4
8         ANTIOQUIA           ANZA                 9
9         ANTIOQUIA       APARTADO                79

Top 10 municipios con más fallecidos:
    Departamento_norm Municipio_norm  total_fallecidos
149         BOGOTA DC         BOGOTA              2429
967   VALLE DEL CAUCA           CALI              1300
64          ANTIOQUIA       MEDELLIN              1081
157           BOLIVAR      CARTAGENA               567
127      

## 6. Cruce de las dos fuentes

Ahora ambas fuentes están al mismo nivel (una fila por Departamento + Municipio). Hacemos el merge con `how="left"` para conservar **todas** las filas de la Fuente 1 (los 316 tramos), aunque algunos municipios no tengan fallecidos registrados.

In [37]:
# Merge por Departamento + Municipio
resultado = pd.merge(
    sectores,
    fallecidos_por_municipio,
    on=["Departamento_norm", "Municipio_norm"],
    how="left"
)

print("=" * 60)
print("RESULTADO DEL CRUCE")
print("=" * 60)
print(f"\nShape del resultado: {resultado.shape}")
print(f"\nPrimeras 5 filas:")
print(resultado.head())

RESULTADO DEL CRUCE

Shape del resultado: (316, 16)

Primeras 5 filas:
    ID_MT       ENTIDAD  GiZScore  Fallecidos  GiPValue  \
0   4G013           ANI  1.771262          11  0.076517   
1   4G013           ANI  3.818008          23  0.000135   
2   4G013           ANI  2.796662           1  0.005163   
3   4G013           ANI  1.725132           1  0.084504   
4  <Null>  INVIAS-OTROS  1.725132          13  0.084504   

                     Tramo                         Nombre   Latitud  \
0   Bogotá - Villavicencio  IP  –  Chirajara - Fundadores  4.452987   
1   Bogotá - Villavicencio  IP  –  Chirajara - Fundadores  4.228251   
2   Bogotá - Villavicencio  IP  –  Chirajara - Fundadores  4.203601   
3   Bogotá - Villavicencio  IP  –  Chirajara - Fundadores  4.121453   
4  Granada - Villavicencio                            NaN  4.115757   

    Longitud  PR      Municipio  Departamento  divipola Departamento_norm  \
0 -74.046677  10       CHIPAQUE  CUNDINAMARCA     25178      CUNDINAMA

## 7. VERIFICACIÓN OBLIGATORIA

Según la Sesión 4 y la rúbrica del Corte 1, después del merge hay que verificar tres cosas:

1. **El número de filas no cambió** (316 antes → 316 después)
2. **Ver filas sin pareja** (tramos cuyo municipio no apareció en la Fuente 2)
3. **Revisar duplicados** en el identificador del tramo

In [38]:
# VERIFICACIÓN 1: Número de filas antes y después
filas_antes = len(sectores)
filas_despues = len(resultado)

print("=" * 60)
print("VERIFICACIÓN 1: NÚMERO DE FILAS")
print("=" * 60)
print(f"Filas antes del merge:   {filas_antes}")
print(f"Filas después del merge: {filas_despues}")

if filas_antes == filas_despues:
    print("\n✅ CORRECTO: el merge no duplicó ni perdió filas.")
else:
    print(f"\n❌ PROBLEMA: diferencia de {filas_despues - filas_antes} filas.")

VERIFICACIÓN 1: NÚMERO DE FILAS
Filas antes del merge:   316
Filas después del merge: 316

✅ CORRECTO: el merge no duplicó ni perdió filas.


In [39]:
# VERIFICACIÓN 2: Filas sin pareja
sin_pareja = resultado[resultado["total_fallecidos"].isna()]

print("=" * 60)
print("VERIFICACIÓN 2: FILAS SIN PAREJA")
print("=" * 60)
print(f"Tramos sin fallecidos registrados en su municipio: {len(sin_pareja)}")
print(f"Porcentaje: {len(sin_pareja) / len(resultado) * 100:.1f}%")
print(f"\nPrimeros 10 tramos sin pareja:")
print(sin_pareja[["Departamento", "Municipio", "Tramo", "Fallecidos"]].head(10))

VERIFICACIÓN 2: FILAS SIN PAREJA
Tramos sin fallecidos registrados en su municipio: 25
Porcentaje: 7.9%

Primeros 10 tramos sin pareja:
           Departamento         Municipio                 Tramo  Fallecidos
55            SANTANDER  SABANA DE TORRES  Barranca - Aguachica          17
56            SANTANDER  SABANA DE TORRES  Barranca - Aguachica          12
74                CESAR        RIO DE ORO  Barranca - Aguachica          11
75   NORTE DE SANTANDER          EL ZULIA                   NaN           4
76   NORTE DE SANTANDER          EL ZULIA       El Zulia-Cúcuta          24
120     VALLE DEL CAUCA              BUGA     Buga- La Victoria          14
121     VALLE DEL CAUCA              BUGA     Buga- La Victoria          13
122     VALLE DEL CAUCA              BUGA     Buga- La Victoria           6
123     VALLE DEL CAUCA              BUGA     Buga- La Victoria           7
124     VALLE DEL CAUCA              BUGA     Buga- La Victoria           8


In [40]:
# VERIFICACIÓN 3: Duplicados en el identificador del tramo
duplicados_id_mt = sectores["ID_MT"].duplicated().sum()
duplicados_tramo = sectores.duplicated(subset=["Tramo", "Latitud", "Longitud"]).sum()

print("=" * 60)
print("VERIFICACIÓN 3: DUPLICADOS")
print("=" * 60)
print(f"ID_MT duplicados: {duplicados_id_mt}")
print(f"Tramos duplicados (por Tramo+Lat+Long): {duplicados_tramo}")

if duplicados_id_mt == 0:
    print("\n✅ No hay ID_MT duplicados.")
else:
    print(f"\n⚠️ Hay {duplicados_id_mt} ID_MT repetidos (se corrigen en la Unidad 3).")

VERIFICACIÓN 3: DUPLICADOS
ID_MT duplicados: 291
Tramos duplicados (por Tramo+Lat+Long): 4

⚠️ Hay 291 ID_MT repetidos (se corrigen en la Unidad 3).


## 8. Diccionario de datos del dataset cruzado

A continuación se documentan las columnas principales del dataset resultante del cruce.

In [41]:
# Construir diccionario de datos
diccionario = pd.DataFrame([
    # Columnas de la Fuente 1 (Sectores Críticos)
    {"columna": "ID_MT", "fuente": "F1 — Sectores Críticos", "tipo": "Texto", "significado": "Identificador de la vía según el Ministerio de Transporte"},
    {"columna": "ENTIDAD", "fuente": "F1 — Sectores Críticos", "tipo": "Texto", "significado": "Entidad a cargo de la vía (ANI, INVIAS, DEPTO, CIUDAD)"},
    {"columna": "GiZScore", "fuente": "F1 — Sectores Críticos", "tipo": "Número", "significado": "Calificación del tramo según análisis espacial (Getis-Ord Gi*)"},
    {"columna": "Fallecidos", "fuente": "F1 — Sectores Críticos", "tipo": "Número", "significado": "Cantidad de fallecidos registrados en ese tramo (2015–2019)"},
    {"columna": "GiPValue", "fuente": "F1 — Sectores Críticos", "tipo": "Número", "significado": "Probabilidad de que el patrón espacial del tramo sea aleatorio"},
    {"columna": "Tramo", "fuente": "F1 — Sectores Críticos", "tipo": "Texto", "significado": "Identificación del tramo vial (máximo 10 km)"},
    {"columna": "Nombre", "fuente": "F1 — Sectores Críticos", "tipo": "Texto", "significado": "Nombre de la vía o corredor"},
    {"columna": "Latitud", "fuente": "F1 — Sectores Críticos", "tipo": "Número", "significado": "Coordenada geográfica norte del tramo"},
    {"columna": "Longitud", "fuente": "F1 — Sectores Críticos", "tipo": "Número", "significado": "Coordenada geográfica este del tramo"},
    {"columna": "PR", "fuente": "F1 — Sectores Críticos", "tipo": "Texto", "significado": "Punto de referencia en la vía"},
    {"columna": "Municipio", "fuente": "F1 — Sectores Críticos", "tipo": "Texto", "significado": "Municipio donde se ubica el tramo"},
    {"columna": "Departamento", "fuente": "F1 — Sectores Críticos", "tipo": "Texto", "significado": "Departamento donde se ubica el tramo"},
    {"columna": "divipola", "fuente": "F1 — Sectores Críticos", "tipo": "Texto", "significado": "Código de división política (DANE)"},
    # Columnas de la Fuente 2 (Fallecidos agrupados)
    {"columna": "Departamento_norm", "fuente": "F1 + F2 (clave de cruce)", "tipo": "Texto", "significado": "Departamento normalizado (mayúsculas, sin tildes) usado para el merge"},
    {"columna": "Municipio_norm", "fuente": "F1 + F2 (clave de cruce)", "tipo": "Texto", "significado": "Municipio normalizado (mayúsculas, sin tildes) usado para el merge"},
    {"columna": "total_fallecidos", "fuente": "F2 — Fallecidos agrupados", "tipo": "Número", "significado": "Total de personas fallecidas en siniestros viales en ese municipio (2021–2024)"},
])

# Mostrar diccionario
print("=" * 60)
print("DICCIONARIO DE DATOS DEL DATASET CRUZADO")
print("=" * 60)
print(f"Total de columnas documentadas: {len(diccionario)}")
print()
for _, row in diccionario.iterrows():
    print(f"• {row['columna']:<25} | {row['fuente']:<30} | {row['tipo']}")

DICCIONARIO DE DATOS DEL DATASET CRUZADO
Total de columnas documentadas: 16

• ID_MT                     | F1 — Sectores Críticos         | Texto
• ENTIDAD                   | F1 — Sectores Críticos         | Texto
• GiZScore                  | F1 — Sectores Críticos         | Número
• Fallecidos                | F1 — Sectores Críticos         | Número
• GiPValue                  | F1 — Sectores Críticos         | Número
• Tramo                     | F1 — Sectores Críticos         | Texto
• Nombre                    | F1 — Sectores Críticos         | Texto
• Latitud                   | F1 — Sectores Críticos         | Número
• Longitud                  | F1 — Sectores Críticos         | Número
• PR                        | F1 — Sectores Críticos         | Texto
• Municipio                 | F1 — Sectores Críticos         | Texto
• Departamento              | F1 — Sectores Críticos         | Texto
• divipola                  | F1 — Sectores Críticos         | Texto
• Departamento_norm  

## 9. Resumen final

El dataset cruzado tiene **316 filas** (una por tramo crítico) y contiene tanto la información geográfica del tramo como el total de fallecidos en su municipio durante 2021–2024. Este dataset es la base para el análisis exploratorio (EDA) de la Sesión 4 y el resto del proyecto integrador.

In [42]:
# Guardar el dataset cruzado para las siguientes fases
resultado.to_csv("dataset_cruzado_siniestralidad.csv", index=False, encoding="utf-8")

print("Dataset cruzado guardado como: dataset_cruzado_siniestralidad.csv")
print(f"Shape final: {resultado.shape}")
print(f"\nColumnas del dataset final:")
for col in resultado.columns:
    print(f"  - {col}")

Dataset cruzado guardado como: dataset_cruzado_siniestralidad.csv
Shape final: (316, 16)

Columnas del dataset final:
  - ID_MT
  - ENTIDAD
  - GiZScore
  - Fallecidos
  - GiPValue
  - Tramo
  - Nombre
  - Latitud
  - Longitud
  - PR
  - Municipio
  - Departamento
  - divipola
  - Departamento_norm
  - Municipio_norm
  - total_fallecidos


In [43]:
# Verificación 2 (ampliada): cuantificar filas sin pareja
sin_pareja = resultado[resultado["total_fallecidos"].isna()]
print(f"Tramos sin pareja: {len(sin_pareja)} de {len(resultado)} ({len(sin_pareja)/len(resultado)*100:.1f}%)")
print(f"\nMunicipios afectados:")
print(sin_pareja[["Departamento", "Municipio", "Tramo"]].to_string())

Tramos sin pareja: 25 de 316 (7.9%)

Municipios afectados:
           Departamento               Municipio                           Tramo
55            SANTANDER        SABANA DE TORRES            Barranca - Aguachica
56            SANTANDER        SABANA DE TORRES            Barranca - Aguachica
74                CESAR              RIO DE ORO            Barranca - Aguachica
75   NORTE DE SANTANDER                EL ZULIA                             NaN
76   NORTE DE SANTANDER                EL ZULIA                 El Zulia-Cúcuta
120     VALLE DEL CAUCA                    BUGA               Buga- La Victoria
121     VALLE DEL CAUCA                    BUGA               Buga- La Victoria
122     VALLE DEL CAUCA                    BUGA               Buga- La Victoria
123     VALLE DEL CAUCA                    BUGA               Buga- La Victoria
124     VALLE DEL CAUCA                    BUGA               Buga- La Victoria
125     VALLE DEL CAUCA                    BUGA              

**Hallazgo:** 25 de los 316 tramos (7.9%) quedaron sin pareja tras el merge,
porque el nombre del municipio está escrito de forma distinta entre las dos
fuentes. Ejemplos concretos:

- La F1 dice `CARTAGENA DE INDIAS`, la F2 probablemente dice `CARTAGENA`.
- La F1 dice `PATÍA (El Bordo)`, la F2 probablemente dice `EL BORDO` o `PATIA`.
- La F1 dice `SANTANDER DE QUILICHAO`, la F2 probablemente dice `QUILICHAO`.
- La F1 dice `CIÉNAGA DE ORO`, la F2 probablemente dice `CIENAGA DE ORO`.
- La F1 dice `BUGA`, la F2 probablemente dice `GUADALAJARA DE BUGA` (nombre oficial).

Esto es un problema conocido al cruzar fuentes oficiales: los nombres de
municipios no están estandarizados entre entidades.

**Mitigación para la Unidad 3:** En la fase de limpieza se usará el código
`divipola` del DANE como clave de cruce, en lugar del nombre del municipio.
El divipola es un identificador numérico único que no cambia entre fuentes.
Adicionalmente, se construirá un diccionario de equivalencias municipio → divipola
para recuperar las filas que hoy quedan sin pareja.

**Impacto actual:** El 7.9% de los tramos no tiene dato de fallecidos 2021-2024,
pero conservan toda su información geográfica y de fallecidos 2015-2019 de la F1.
El 92.1% restante sí cruzó correctamente, lo que permite un análisis robusto.